In [1]:
from datasets import load_dataset
import pandas as pd


In [ ]:
# ds = load_dataset("TimKoornstra/financial-tweets-sentiment") # import TIM dataset 

# df = pd.DataFrame(ds['train']) # convert to pandas dataframe

# df.to_csv('Tim_financial_tweets_sentiment.csv', index=False) # save to csv


In [4]:
df_sent_train = pd.read_csv('sent_train.csv') 
df_sent_valid = pd.read_csv('sent_valid.csv')


In [5]:
df_sent_train.head()

,text,label
0,$BYND - JPMorgan reels in expectations on Beyo...,0
1,$CCL $RCL - Nomura points to bookings weakness...,0
2,"$CX - Cemex cut at Credit Suisse, J.P. Morgan ...",0
3,$ESS: BTIG Research cuts to Neutral https://t....,0
4,$FNKO - Funko slides after Piper Jaffray PT cu...,0


In [7]:
df_sent_valid.head()

,text,label
0,$ALLY - Ally Financial pulls outlook https://t...,0
1,"$DELL $HPE - Dell, HPE targets trimmed on comp...",0
2,$PRTY - Moody's turns negative on Party City h...,0
3,$SAN: Deutsche Bank cuts to Hold,0
4,$SITC: Compass Point cuts to Sell,0


In [12]:
df_tim = pd.read_csv('Tim_financial_tweets_sentiment.csv') # import TIM dataset
df_tim.head()

,tweet,sentiment,url
0,$BYND - JPMorgan reels in expectations on Beyo...,2,https://huggingface.co/datasets/zeroshot/twitt...
1,$CCL $RCL - Nomura points to bookings weakness...,2,https://huggingface.co/datasets/zeroshot/twitt...
2,"$CX - Cemex cut at Credit Suisse, J.P. Morgan ...",2,https://huggingface.co/datasets/zeroshot/twitt...
3,$ESS: BTIG Research cuts to Neutral https://t....,2,https://huggingface.co/datasets/zeroshot/twitt...
4,$FNKO - Funko slides after Piper Jaffray PT cu...,2,https://huggingface.co/datasets/zeroshot/twitt...


In [14]:
df_tim.drop('url', axis=1, inplace=True) # drop url column
df_tim.head()

,tweet,sentiment
0,$BYND - JPMorgan reels in expectations on Beyo...,2
1,$CCL $RCL - Nomura points to bookings weakness...,2
2,"$CX - Cemex cut at Credit Suisse, J.P. Morgan ...",2
3,$ESS: BTIG Research cuts to Neutral https://t....,2
4,$FNKO - Funko slides after Piper Jaffray PT cu...,2


In [28]:
len(df_tim)

38091

In [15]:
df_sent_train.rename({'label': 'sentiment'}, axis=1, inplace=True) # rename label to sentiment
df_sent_train.rename({'text': 'tweet'}, axis=1, inplace=True) # rename text to tweet

df_sent_valid.rename({'label': 'sentiment'}, axis=1, inplace=True) # rename label to sentiment
df_sent_valid.rename({'text': 'tweet'}, axis=1, inplace=True) # rename text to tweet




In [16]:
df_sent_train.head()

,tweet,sentiment
0,$BYND - JPMorgan reels in expectations on Beyo...,0
1,$CCL $RCL - Nomura points to bookings weakness...,0
2,"$CX - Cemex cut at Credit Suisse, J.P. Morgan ...",0
3,$ESS: BTIG Research cuts to Neutral https://t....,0
4,$FNKO - Funko slides after Piper Jaffray PT cu...,0


In [20]:
len(df_sent_train)

9543

In [17]:
df_sent_valid.head()

,tweet,sentiment
0,$ALLY - Ally Financial pulls outlook https://t...,0
1,"$DELL $HPE - Dell, HPE targets trimmed on comp...",0
2,$PRTY - Moody's turns negative on Party City h...,0
3,$SAN: Deutsche Bank cuts to Hold,0
4,$SITC: Compass Point cuts to Sell,0


In [21]:
len(df_sent_valid)

2388

In [18]:
df_combined_sent = pd.concat([df_sent_train, df_sent_valid], ignore_index=True) # combine train and valid datasets
df_combined_sent.head()

,tweet,sentiment
0,$BYND - JPMorgan reels in expectations on Beyo...,0
1,$CCL $RCL - Nomura points to bookings weakness...,0
2,"$CX - Cemex cut at Credit Suisse, J.P. Morgan ...",0
3,$ESS: BTIG Research cuts to Neutral https://t....,0
4,$FNKO - Funko slides after Piper Jaffray PT cu...,0


In [19]:
print(len(df_combined_sent))

11931


In [22]:
df_combined_sent.dropna(subset=['sentiment', 'tweet'], inplace=True) # drop rows with missing values in sentiment and tweet columns
df_combined_sent.reset_index(drop=True, inplace=True) # reset index
len(df_combined_sent)

11931

In [23]:
df_combined_sent['sentiment'] = df_combined_sent['sentiment'].replace({0: 2, 2: 0})

# Verify the changes
print(df_combined_sent['sentiment'].value_counts())
df_combined_sent.head()

sentiment
0    7744
1    2398
2    1789
Name: count, dtype: int64


,tweet,sentiment
0,$BYND - JPMorgan reels in expectations on Beyo...,2
1,$CCL $RCL - Nomura points to bookings weakness...,2
2,"$CX - Cemex cut at Credit Suisse, J.P. Morgan ...",2
3,$ESS: BTIG Research cuts to Neutral https://t....,2
4,$FNKO - Funko slides after Piper Jaffray PT cu...,2


In [25]:
df_final = pd.concat([df_combined_sent, df_tim], ignore_index=True) # combine sentiment and TIM datasets
# Verify the combined DataFrame
print(len(df_final))


50022


In [27]:
df_final.head()

,tweet,sentiment
0,$BYND - JPMorgan reels in expectations on Beyo...,2
1,$CCL $RCL - Nomura points to bookings weakness...,2
2,"$CX - Cemex cut at Credit Suisse, J.P. Morgan ...",2
3,$ESS: BTIG Research cuts to Neutral https://t....,2
4,$FNKO - Funko slides after Piper Jaffray PT cu...,2


In [30]:
from sklearn.model_selection import train_test_split

train_data, temp_data = train_test_split(df_final, test_size=0.2, random_state=42, stratify=df_final['sentiment'])
valid_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42, stratify=temp_data['sentiment'])

print(f"Train size: {len(train_data)}")
print(f"Validation size: {len(valid_data)}")
print(f"Test size: {len(test_data)}")


Train size: 40017
Validation size: 5002
Test size: 5003


In [31]:
train_data.to_csv('sentiment_train_data.csv', index=False) # save train data to csv
valid_data.to_csv('sentiment_valid_data.csv', index=False) # save valid data to csv
test_data.to_csv('sentiment_test_data.csv', index=False) # save test data to csv